# 01 — Data Collection: Financial News Scraping

This notebook scrapes financial news headlines for all **30 Dow Jones Industrial Average (DJIA)** component stocks from two sources:

| Source | Method | Why |
|---|---|---|
| **Markets Insider** | `requests` + `BeautifulSoup` | Static HTML — fast and lightweight |
| **Financial Times** | `Selenium` (headless Chrome) | JavaScript-rendered pages require a real browser |

**Output:** `../data/correlationimp.csv` — a combined dataset of `(datetime, ticker, headline, source, link)` rows covering **2022–2023**.

> ⚠️ **Note:** Running this notebook requires an active internet connection and, for the FT section, a compatible `chromedriver.exe` placed in the same directory (or on your system PATH). The pre-scraped dataset is already included in the repository as `../data/correlationimp.csv`.

## Section 1 — Markets Insider (BeautifulSoup)

Markets Insider serves static HTML pages, making `requests` + `BeautifulSoup` an efficient choice.

### Strategy
- For each of the 30 DJIA tickers, iterate through up to 400 pages of the Markets Insider news feed.
- Extract the article **publication datetime**, **ticker code**, **headline title**, **source publication**, and **article link** from each `div.latest-news__story` element.
- Accumulate results in a list and construct a single DataFrame at the end (avoids the performance cost of repeated `pd.concat` inside the loop).
- Save to `../data/mi_sent.csv`.

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

# ── DJIA component tickers ───────────────────────────────────────────────────
DJIA_TICKERS = [
    "MMM", "AXP", "AMGN", "AAPL", "BA",  "CAT", "CVX", "CSCO", "KO",
    "DOW", "GS",  "HD",   "HON",  "IBM", "INTC", "JNJ", "JPM",  "MCD",
    "MRK", "MSFT","NKE",  "PG",   "CRM", "TRV", "UNH", "VZ",   "V",
    "WBA", "WMT", "DIS"
]

MAX_PAGES   = 400   # Upper bound on paginated results per ticker
SLEEP_SECS  = 0.5   # Polite delay between requests

# ── Scraping ─────────────────────────────────────────────────────────────────
rows = []           # Accumulate rows here; build DataFrame once at the end
article_count = 0

for ticker in DJIA_TICKERS:
    for page in range(1, MAX_PAGES + 1):
        url = f"https://markets.businessinsider.com/news/{ticker}-stock?p={page}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "lxml")
        articles = soup.find_all("div", class_="latest-news__story")

        # Stop paginating when the page returns no articles
        if not articles:
            break

        for article in articles:
            try:
                datetime_val = article.find("time",  class_="latest-news__date").get("datetime")
                title        = article.find("a",     class_="news-link").text.strip()
                source       = article.find("span",  class_="latest-news__source").text.strip()
                link         = article.find("a",     class_="news-link").get("href")
                rows.append([datetime_val, ticker, title, source, link])
                article_count += 1
            except AttributeError:
                # Skip malformed / incomplete article cards
                continue

        time.sleep(SLEEP_SECS)

# ── Build DataFrame and save ──────────────────────────────────────────────────
mi_df = pd.DataFrame(rows, columns=["datetime", "code", "title", "source", "link"])
mi_df.to_csv("../data/mi_sent.csv", index=False)

print(f"Markets Insider: {article_count:,} articles scraped across {len(DJIA_TICKERS)} tickers.")
mi_df.head()

## Section 2 — Financial Times (Selenium)

The Financial Times renders its search results via JavaScript, so `requests` alone cannot retrieve article listings. **Selenium** with a headless Chrome browser is used instead.

### Prerequisites
1. Google Chrome installed on your machine.
2. A compatible `chromedriver.exe` in the same folder as this notebook, or on your system `PATH`.
   - Download from https://chromedriver.chromium.org/downloads (match your Chrome version).

### Strategy
- For each DJIA company name, build a quoted search URL targeting FT results from **2022-01-01 to 2023-12-31**.
- Use `WebDriverWait` to confirm the page has loaded before parsing.
- A retry helper (`fetch_url`) handles transient network failures with exponential back-off.
- A random sleep between requests mimics human browsing and avoids rate-limiting.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# ── DJIA company names (used as FT search queries) ────────────────────────────
DJIA_COMPANIES = [
    "3M", "American Express", "Amgen", "Apple", "Boeing",
    "Caterpillar", "Chevron", "Cisco", "Coca-Cola", "Dow",
    "Goldman Sachs", "Home Depot", "Honeywell", "IBM", "Intel",
    "Johnson & Johnson", "JPMorgan Chase", "McDonald's", "Merck",
    "Microsoft", "NIKE", "Procter & Gamble", "Salesforce", "Travelers",
    "UnitedHealth", "Verizon", "Visa", "Walgreens", "Walmart", "Disney"
]

DATE_FROM = "2022-01-01"
DATE_TO   = "2023-12-31"


def fetch_url(driver: webdriver.Chrome, url: str, retries: int = 3, backoff_factor: float = 0.3):
    """Navigate to *url* with retry logic and exponential back-off.
    
    Returns the page source HTML on success, or None if all retries fail.
    Waits for the `o-teaser__heading` class to confirm the FT search page loaded.
    """
    for attempt in range(retries):
        try:
            driver.get(url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "o-teaser__heading"))
            )
            return driver.page_source
        except (TimeoutException, WebDriverException) as exc:
            print(f"  Attempt {attempt + 1} failed: {exc}")
            time.sleep(backoff_factor * (2 ** attempt))
    return None


# ── Configure headless Chrome ─────────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-infobars")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
)

service = Service("chromedriver.exe")
driver  = webdriver.Chrome(service=service, options=chrome_options)

# ── Scraping ─────────────────────────────────────────────────────────────────
rows = []
article_count = 0

for company in DJIA_COMPANIES:
    query = f'"{company}"'
    url = (
        f"https://www.ft.com/search?q={query}"
        f"&page=1&dateTo={DATE_TO}&dateFrom={DATE_FROM}"
        "&sort=relevance&expandRefinements=true&isFirstView=false"
    )

    html = fetch_url(driver, url)
    if html is None:
        print(f"  Skipping {company} — failed to retrieve page.")
        continue

    soup = BeautifulSoup(html, "lxml")
    headings = soup.find_all("div", class_="o-teaser__heading")

    # Capture pagination metadata (useful for debugging coverage)
    pagination = soup.find("span", class_="search-pagination__page")
    page_info = pagination.get_text(strip=True) if pagination else "N/A"

    for heading in headings:
        a_tag = heading.find("a", class_="js-teaser-heading-link")
        if not a_tag:
            continue

        title = " ".join(a_tag.stripped_strings)

        # The timestamp lives in the sibling `.o-teaser__timestamp` block
        timestamp_div = heading.find_next("div", class_="o-teaser__timestamp")
        if not timestamp_div:
            continue

        time_tag = timestamp_div.find("time", class_="o-teaser__timestamp-date")
        if not time_tag:
            continue

        date = time_tag.get_text(strip=True)
        rows.append([title, date, query, page_info])
        article_count += 1

    # Polite delay to avoid rate-limiting
    time.sleep(random.uniform(1.0, 3.0))

driver.quit()

# ── Build DataFrame ───────────────────────────────────────────────────────────
ft_df = pd.DataFrame(rows, columns=["title", "date", "code", "page_info"])
print(f"Financial Times: {article_count:,} articles scraped across {len(DJIA_COMPANIES)} companies.")
ft_df.head()

## Section 3 — Combine and Export

The two sources share the key fields `(datetime, code, title)`. We align column names and concatenate into a single file:
**`../data/correlationimp.csv`** — the input dataset for Notebook 02.

In [ ]:
# Rename FT columns to match Markets Insider schema
ft_df_aligned = ft_df[["date", "code", "title"]].rename(columns={"date": "datetime"})

# Markets Insider already has (datetime, code, title, source, link)
mi_df_aligned = mi_df[["datetime", "code", "title"]]

# Combine
combined_df = pd.concat([mi_df_aligned, ft_df_aligned], ignore_index=True)

# Save
combined_df.to_csv("../data/correlationimp.csv", index=False)

print(f"Combined dataset: {len(combined_df):,} articles saved to ../data/correlationimp.csv")
combined_df.head(10)